# Train and Validate
* Train a model across the specified training sites and sampling approach
* Validate the model across test data

In [ ]:
import pathlib
import numpy
import dask.distributed
import pandas
import time

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling
import plotting

%load_ext autoreload
%autoreload 2

In [ ]:
cluster = dask.distributed.LocalCluster(
    threads_per_worker=1,   n_workers=4,                                 
    config={
        "distributed.worker.memory.target": 0.80, 
        "distributed.worker.memory.spill": 0.85,
        "distributed.worker.memory.pause": 0.90
    }
)
client = dask.distributed.Client(cluster)
display(client)

# Values to edit

In [ ]:
uav_classes_to_ignore = ['Shadow', 'Glare']

In [ ]:
# Mappings of Satellite classes to consider from UAV classes - 98% purity
satellite_classes_all = {'Seagrass': 1, 'Seagrass submerged': 2, 'Gracilaria': 3, 'Gracilaria submerged': 4, 
                           'Ulva': 5, 'Submerged vegetation': 9, 'Microphytobenthos': 10, 'Satmarsh': 14,
                           'Unvegetated': 15, 'Water': 16, 'Terrestrial': 18, 'Rock': 19, 'Mixed': 22,}
satellite_classes_all = {'Seagrass': 0, 'Seagrass submerged': 1, 'Gracilaria': 2, 'Gracilaria submerged': 3, 
                           'Ulva': 4, 'Submerged vegetation': 5, 'Microphytobenthos': 6, 'Satmarsh': 7,
                           'Unvegetated': 8, 'Water': 9, 'Terrestrial': 10, 'Rock': 11, 'Mixed': 12,}
satellite_classes_from_uav_classes_all = {
    'Seagrass': ['Seagrass'],
    'Gracilaria': ['Gracilaria'],
    'Ulva': ['Ulva', 'Ulva mats'],
    'Satmarsh': ['Saltmarsh', ],
    'Terrestrial': ['Terrestrial'],
    'Unvegetated': ['Unvegetated'],                            
    'Water': ['Water'],
    'Mixed': ['Cystophora', 'Hormosira','Brown algae mixed', 'Green algae mixed', 
          'Filamentous brown algae', 'Red algae mixed'],
    'Submerged vegetation': ['Submerged vegetation'],
    'Gracilaria submerged': ['Gracilaria submerged'],
    'Seagrass submerged': ['Seagrass submerged'],
    'Rock': ['Rock'],
    'Microphytobenthos': ['Microphytobenthos'], 
}

In [ ]:
# Mappings of Satellite classes to consider from UAV classes - 98% purity
satellite_classes_grouped = {'Seagrass': 1, 'Seagrass submerged': 2, 'Gracilaria': 3,  
                           'Ulva': 5, 'Submerged vegetation': 9, 'Microphytobenthos': 10, 'Satmarsh': 14,
                           'Unvegetated': 15, 'Water': 16, 'Terrestrial': 18, 'Rock': 19, 'Mixed': 22,}
satellite_classes_from_uav_classes_grouped = {
    'Seagrass': ['Seagrass'],
    'Gracilaria': ['Gracilaria', 'Gracilaria submerged'],
    'Ulva': ['Ulva', 'Ulva mats'],
    'Satmarsh': ['Saltmarsh', ],
    'Terrestrial': ['Terrestrial'],
    'Unvegetated': ['Unvegetated'],                            
    'Water': ['Water'],
    'Mixed': ['Cystophora', 'Hormosira','Brown algae mixed', 'Green algae mixed', 
          'Filamentous brown algae', 'Red algae mixed'],
    'Submerged vegetation': ['Submerged vegetation'],
    'Seagrass submerged': ['Seagrass submerged'],
    'Rock': ['Rock'],
    'Microphytobenthos': ['Microphytobenthos'], 
}

In [ ]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua",
                       "Purau", "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26",
                       "Paremata_Nov25", "Paremata_Feb26", "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26",
                       "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

### Make sure you change the `model_file` name when experimenting

In [ ]:
sample_method = "sampling_2"
method_2_threshold = .5 # .5 .75
max_cloud_cover = 5 # percentage
low_tide_delta_hrs = 1
low_tide_delta_mins = 0
test_threshold = 0.1

num_epoch = 150
tile_size = 128 # 64 128
tile_stride = 64 # None #64, 32

filtered=True
filtered_label = "filtered_" if filtered else ""

In [ ]:
training_sites = all_training_sites # training_sites_SI all_training_sites ["Duvauchelle"]

models_path = utils.get_models_path(
    sample_method=sample_method,
    method_2_threshold=method_2_threshold,
    low_tide_delta_hrs=low_tide_delta_hrs,
    low_tide_delta_mins=low_tide_delta_mins,
    max_cloud_cover=max_cloud_cover,
    use_threshold=False,)
model_description = f"UN_model_{filtered_label}dates_13_class_{int(test_threshold*100)}_percent_test_{num_epoch}_epochs_{tile_size}_tiling_{tile_stride}_stride"  
# f"UN_model_all_trained_13_class_{int(test_threshold*100)}_percent_test_{num_epoch}_epochs_{tile_size}_tiling_{tile_stride}_stride"
models_path = models_path / model_description
(models_path / "diffuser_unet").mkdir(exist_ok=True, parents=True)

satellite_classes = satellite_classes_all
satellite_classes_from_uav_classes = satellite_classes_from_uav_classes_all

# Cells to run
* Train and save model
* Review model
  * satellite bands of each UAV class
  * satellite bands of each satellite class
  * importance of satellite bands in trained model
* Validation
  * Predict excluded site
  * Plot confusion matrix comparing prediction to UAV classifications

In [ ]:
data_path = utils.get_data_path()
utils.create_data_folders()
uav_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
sample_folder = utils.get_samples_path(
    sample_method=sample_method,
    method_2_threshold=method_2_threshold,
    low_tide_delta_hrs=low_tide_delta_hrs,
    low_tide_delta_mins=low_tide_delta_mins,
    max_cloud_cover=max_cloud_cover,
    filtered=filtered,
    use_threshold=False,
)

validation_path = utils.get_validation_path(
    sample_method=sample_method,
    method_2_threshold=method_2_threshold,
    low_tide_delta_hrs=low_tide_delta_hrs,
    low_tide_delta_mins=low_tide_delta_mins,
    max_cloud_cover=max_cloud_cover,
    filtered=filtered,
    use_threshold=False,)
test_prediction_file = validation_path / model_description / "diffuser_unet" / f"{int(test_threshold*100)}_percent_prediction.npy"
test_prediction_file.parent.mkdir(exist_ok=True, parents=True)

### Train and save model

In [ ]:
models_path / "diffuser_unet"

In [ ]:
if not (models_path / "diffuser_unet" / "training_data_summary.csv").exists():

    tiles_array, labels_array = training.load_samples_and_tile(training_sites=training_sites,low_tide_delta_hrs=low_tide_delta_hrs,
                                                 low_tide_delta_mins=low_tide_delta_mins, max_cloud_cover=max_cloud_cover,
                                                 method_2_threshold=method_2_threshold,tile_size=tile_size,stride=tile_stride, filtered=filtered,)
    train_tiles, train_labels, test_tiles, test_labels = training.randomise_tiles_to_test_and_training(
        tiles=tiles_array, labels=labels_array, test_threshold=test_threshold
    )
    scl_channel = sentinel2.BANDS.index("SCL")
    train_scl_array = train_tiles[:, scl_channel, :, :]
    test_scl_array = test_tiles[:, scl_channel, :, :]
    train_labels = training.map_satellite_ids_into_labels_array(
        labels_array=train_labels,
        uav_labels_file=uav_labels_file,
        uav_classes_to_ignore=uav_classes_to_ignore,
        satellite_classes=satellite_classes,
        satellite_from_uav_classes=satellite_classes_from_uav_classes,
        scl_array=train_scl_array,
        drop_scl_classes_to_ignore=True,
)
    test_labels = training.map_satellite_ids_into_labels_array(
        labels_array=test_labels,
        uav_labels_file=uav_labels_file,
        uav_classes_to_ignore=uav_classes_to_ignore,
        satellite_classes=satellite_classes,
        satellite_from_uav_classes=satellite_classes_from_uav_classes,
        scl_array=test_scl_array,
        drop_scl_classes_to_ignore=True,
)
    # Save split test and train dataset
    numpy.save(models_path / "diffuser_unet" / "train_tiles.npy", train_tiles)
    numpy.save(models_path / "diffuser_unet" / "train_labels.npy", train_labels)
    numpy.save(models_path / "diffuser_unet" / "test_tiles.npy", test_tiles)
    numpy.save(models_path / "diffuser_unet" / "test_labels.npy", test_labels)

    # perform training
    start = time.perf_counter()
    monitor_class_ids = {
        class_name: satellite_classes[class_name]
        for class_name in ["Seagrass", "Seagrass submerged", "Gracilaria", "Gracilaria submerged", "Ulva"]
    }
    model, best_model_path = training.train_diffuser_unet_classifier(
        tiles=train_tiles,
        labels=train_labels,
        models_path=models_path,
        num_classes=max(numpy.max(test_labels), numpy.max(train_labels)) + 1,
        epochs=num_epoch,
        val_tiles=test_tiles,
        val_labels=test_labels,
        monitor_class_ids=monitor_class_ids,
        #resume_from_checkpoint=models_path / "diffuser_unet" / "diffuser-unet-epoch=56-val_loss=0.0941.ckpt"  ## uncoment if hot starting
    )
    elapsed = time.perf_counter() - start
    print(f"Elapsed time: {elapsed:.4f} seconds tp fit the model")
    
    # Save a record of the classes considered in training
    satellite_classes_dataframe = pandas.DataFrame.from_dict(satellite_classes, orient='index', columns=['satellite_class_id'])
    satellite_classes_dataframe['uav_class_ids'] = satellite_classes_dataframe.index.map(lambda key: f"{satellite_classes_from_uav_classes[key]}")
    satellite_classes_dataframe.to_csv(models_path / "diffuser_unet" / "class_mappings.csv", index=True)

    # Save a summary of the training data
    training_data_summary = (
        pandas.Series(train_labels.ravel())
        .value_counts()
        .rename_axis("satellite_class_id")
        .reset_index(name="count")
    )
    training_data_summary["satellite_class_name"] = training_data_summary['satellite_class_id'].apply(
        lambda satellite_id: next((key for key, value in satellite_classes.items() if value == int(satellite_id)), None) )
    training_data_summary = training_data_summary[['satellite_class_id', "satellite_class_name", "count"]] # Change column order
    training_data_summary.to_csv(models_path / "diffuser_unet" / "training_data_summary.csv")
    
else:
    print(f"Model '{models_path.name}' already exists. Delete if you want to recreate it.")
    training_data_summary = pandas.read_csv(models_path / "diffuser_unet" / "training_data_summary.csv")
    best_model_path = models_path / "name_here"
    train_tiles = numpy.load(models_path / "diffuser_unet" / "train_tiles.npy")
    train_labels = numpy.load(models_path / "diffuser_unet" / "train_labels.npy")
    test_tiles = numpy.load(models_path / "diffuser_unet" / "test_tiles.npy")
    test_labels = numpy.load(models_path / "diffuser_unet" / "test_labels.npy")
    

training_data_summary

### Review model

In [ ]:
# plot spectra

# Validate
* Predict
* Confusion matrix at UAV resolution
* Confusion matrix at Satellite resolution - take mode of UAV data

In [ ]:
best_model_path

In [ ]:
test_prediction_file

In [ ]:
if not test_prediction_file.exists() or True:
    predictions = training.predict_samples_unet(test_tiles=test_tiles,
                                                checkpoint_file=best_model_path,
                                                weights_only=False)
    numpy.save(test_prediction_file, predictions)
else:
    print(f"Prediction file '{test_prediction_file.name}' already eixsts. Delete if you want to repredict.")
    numpy.save(test_prediction_file, predictions)
    
'''print(
    f"Satellite training classes present: {[key for key, value in satellite_classes.items() if value in predictions['satellite_class_id'].unique()]}. "
    f"Predicted classes present: {[key for key, value in satellite_classes.items() if value in predictions['predicted_class_id'].unique()]}. "
     )'''

plot_filename = test_prediction_file.with_name( f"{test_prediction_file.stem}_confusion_matrix.png")
plot_title = f"Test by {test_threshold*100} percent randomised pixels balenced by class"

training.confusion_matrix_of_pixels(
    truth=numpy.hstack(numpy.hstack(test_labels)),
    predictions=numpy.hstack(numpy.hstack(predictions)),
    satellite_classes=satellite_classes,
    plot_filename=plot_filename,
    plot_title=plot_title
);
